In [ ]:
import numpy as np
import pandas as pd

class InformationGainFeatureSelector:

    def __init__(self, k, target_col, qbins=3):
        self.k = k
        self.target_col = target_col
        self.qbins = qbins
        self.info_gains_ = {}
        self.selected_features_ = []
        
    # Compute entropy of a target variable
    def _entropy(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return -np.sum([p * np.log2(p) for p in probabilities if p > 0])
        
    def _conditional_entropy(self, X_col, y):
        unique_values, counts = np.unique(X_col, return_counts=True)
        conditional_ent = 0
        
        for value, count in zip(unique_values, counts):
            y_subset = y[X_col == value]
            weight = count / len(y)
            conditional_ent += weight * self._entropy(y_subset)
            
        return conditional_ent
        
    # Compute Information Gain for all features
    def _calculate_information_gain(self, df):
        y = df[self.target_col].values
        Hy = self._entropy(y)
        
        info_gains = {}
        
        for col in df.columns:
            if col == self.target_col:
                continue
                
            # Discretize continuous feature into bins using quantiles
            try:
                X_col_discretized = pd.qcut(df[col], q=self.qbins, labels=False)
            except ValueError:
                # Fallback to pure bins if quantiles fail due to many duplicate values
                X_col_discretized = pd.cut(df[col], bins=self.qbins, labels=False, duplicates='drop')
                
            X_col_values = X_col_discretized.values
            info_gains[col] = Hy - self._conditional_entropy(X_col_values, y)
            
        return info_gains
        
    # Fit: calculate IG and rank features
    def fit(self, df):
        if self.target_col is None:
            raise ValueError("target_col must be provided.")
            
        self.info_gains_ = self._calculate_information_gain(df)
        
        # Sort features by IG in descending order
        sorted_features = sorted(
            self.info_gains_,
            key=self.info_gains_.get,
            reverse=True
        )
        
        self.selected_features_ = sorted_features[:self.k]
        
        return self
        
    def transform(self, df):
        return df[self.selected_features_ + [self.target_col]]
        
    def fit_transform(self, df):
        return self.fit(df).transform(df)



y = df_wine['target'].values
print("Entropy of target =", ig_selector._entropy(y))



Entropy of target = 1.5668222768551812


In [28]:
from sklearn.datasets import load_wine
import pandas as pd

# Load wine dataset
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine['target'] = wine.target

# Instantiate InformationGainFeatureSelector
# We will select the top 5 features for demonstration
ig_selector = InformationGainFeatureSelector(k=5, target_col="target")

# Fit the selector
ig_selector.fit(df_wine)

# Get the transformed dataset with only selected features and target
df_transformed = ig_selector.transform(df_wine)
df_transformed.head()


,flavanoids,proline,od280/od315_of_diluted_wines,color_intensity,alcohol,target
0,3.06,1065.0,3.92,5.64,14.23,0
1,2.76,1050.0,3.40,4.38,13.20,0
2,3.24,1185.0,3.17,5.68,13.16,0
3,3.49,1480.0,3.45,7.80,14.37,0
4,2.69,735.0,2.93,4.32,13.24,0


In [29]:
# Display IG scores sorted from highest to lowest
pd.Series(ig_selector.info_gains_).sort_values(ascending=False)


flavanoids                      0.801069
proline                         0.607607
od280/od315_of_diluted_wines    0.588277
color_intensity                 0.583257
alcohol                         0.553560
total_phenols                   0.492992
hue                             0.464107
malic_acid                      0.317008
alcalinity_of_ash               0.289832
proanthocyanins                 0.264757
magnesium                       0.192479
nonflavanoid_phenols            0.185463
ash                             0.092906
dtype: float64